# Day 5 — Python: File Handling (CSV, JSON, TXT)

> **Topics:** `csv` module, `json` module, generators for large files  
> **No pandas** — stdlib only  
> **Run order:** top to bottom, one cell at a time

---
## Setup — Create Sample Files

Run this cell first. It writes three sample files used throughout this notebook:
- `employees.csv` — employee name, dept, salary
- `orders.json` — list of order dicts with status field
- `app.log` — mixed log lines (INFO + ERROR)

In [1]:
import csv, json, os

# ── employees.csv ──────────────────────────────────────────────
employees = [
    ['name', 'dept', 'salary'],
    ['Alice',   'Engineering', '95000'],
    ['Bob',     'Marketing',   '72000'],
    ['Carol',   'Engineering', '105000'],
    ['Dave',    'HR',          '68000'],
    ['Eve',     'Marketing',   '78000'],
    ['Frank',   'Engineering', '88000'],
    ['Grace',   'HR',          '71000'],
    ['Hank',    'Engineering', '92000'],
]
with open('employees.csv', 'w', newline='') as f:
    csv.writer(f).writerows(employees)

# ── orders.json ────────────────────────────────────────────────
orders = [
    {'order_id': 1, 'customer': 'Alice', 'amount': 250.0,  'status': 'completed'},
    {'order_id': 2, 'customer': 'Bob',   'amount': 89.99,  'status': 'pending'},
    {'order_id': 3, 'customer': 'Carol', 'amount': 450.0,  'status': 'completed'},
    {'order_id': 4, 'customer': 'Dave',  'amount': 12.50,  'status': 'cancelled'},
    {'order_id': 5, 'customer': 'Eve',   'amount': 320.0,  'status': 'completed'},
    {'order_id': 6, 'customer': 'Frank', 'amount': 75.0,   'status': 'pending'},
    {'order_id': 7, 'customer': 'Grace', 'amount': 180.0,  'status': 'completed'},
]
with open('orders.json', 'w') as f:
    json.dump(orders, f, indent=2)

# ── app.log ────────────────────────────────────────────────────
log_lines = [
    '2024-01-10 09:00:01 INFO  Server started on port 8080',
    '2024-01-10 09:01:15 INFO  User alice logged in',
    '2024-01-10 09:02:30 ERROR Database connection timeout after 30s',
    '2024-01-10 09:03:05 INFO  Retrying database connection',
    '2024-01-10 09:03:08 INFO  Database connected',
    '2024-01-10 09:04:20 ERROR Failed to process payment for order #1234',
    '2024-01-10 09:05:00 INFO  Cache cleared successfully',
    '2024-01-10 09:06:45 ERROR Disk space below 10% threshold',
    '2024-01-10 09:07:30 INFO  Backup job started',
    '2024-01-10 09:08:00 INFO  Backup job completed',
    '2024-01-10 09:09:12 ERROR API rate limit exceeded for service: payments',
    '2024-01-10 09:10:00 INFO  Scheduled maintenance window starting',
]
with open('app.log', 'w') as f:
    f.write('\n'.join(log_lines) + '\n')

print('Sample files created:')
for fname in ['employees.csv', 'orders.json', 'app.log']:
    print(f'  {fname} — {os.path.getsize(fname)} bytes')

Sample files created:
  employees.csv — 191 bytes
  orders.json — 734 bytes
  app.log — 683 bytes


---
## 1. CSV — `csv.reader` and `csv.DictReader`

In [2]:
# csv.reader — each row is a list of strings
import csv

with open('employees.csv', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)      # pull off the header row
    print('Header:', header)
    for row in reader:
        print(row)             # ['Alice', 'Engineering', '95000']

Header: ['name', 'dept', 'salary']
['Alice', 'Engineering', '95000']
['Bob', 'Marketing', '72000']
['Carol', 'Engineering', '105000']
['Dave', 'HR', '68000']
['Eve', 'Marketing', '78000']
['Frank', 'Engineering', '88000']
['Grace', 'HR', '71000']
['Hank', 'Engineering', '92000']


In [4]:
# csv.DictReader — each row is {column_name: value}
with open('employees.csv', newline='') as f:
    for row in csv.DictReader(f):
        print(row)
        print(f"{row['name']:<10} dept={row['dept']:<15} salary={row['salary']}")

{'name': 'Alice', 'dept': 'Engineering', 'salary': '95000'}
Alice      dept=Engineering     salary=95000
{'name': 'Bob', 'dept': 'Marketing', 'salary': '72000'}
Bob        dept=Marketing       salary=72000
{'name': 'Carol', 'dept': 'Engineering', 'salary': '105000'}
Carol      dept=Engineering     salary=105000
{'name': 'Dave', 'dept': 'HR', 'salary': '68000'}
Dave       dept=HR              salary=68000
{'name': 'Eve', 'dept': 'Marketing', 'salary': '78000'}
Eve        dept=Marketing       salary=78000
{'name': 'Frank', 'dept': 'Engineering', 'salary': '88000'}
Frank      dept=Engineering     salary=88000
{'name': 'Grace', 'dept': 'HR', 'salary': '71000'}
Grace      dept=HR              salary=71000
{'name': 'Hank', 'dept': 'Engineering', 'salary': '92000'}
Hank       dept=Engineering     salary=92000


In [4]:
# Aggregation — average salary per department (no pandas!)
from collections import defaultdict

acc = defaultdict(lambda: {'total': 0.0, 'count': 0})

with open('employees.csv', newline='') as f:
    for row in csv.DictReader(f):
        dept = row['dept']
        acc[dept]['total'] += float(row['salary'])
        acc[dept]['count'] += 1

print('Average salary per department:')
for dept, v in sorted(acc.items()):
    avg = round(v['total'] / v['count'], 2)
    print(f'  {dept:<15} ${avg:>10,.2f}')

Average salary per department:
  Engineering     $ 95,000.00
  HR              $ 69,500.00
  Marketing       $ 75,000.00


In [ ]:
# csv.writer — write rows to a new file
summary = [['dept', 'avg_salary', 'headcount']]
for dept, v in sorted(acc.items()):
    avg = round(v['total'] / v['count'], 2)
    summary.append([dept, avg, v['count']])

with open('dept_summary.csv', 'w', newline='') as f:
    csv.writer(f).writerows(summary)

# Verify by reading back
with open('dept_summary.csv') as f:
    print(f.read())

---
## 2. JSON — `json.load`, `json.dump`, `json.loads`

In [5]:
# json.load — read entire JSON file → Python object
import json

with open('orders.json') as f:
    orders = json.load(f)      # returns a list of dicts

print(f'Total orders: {len(orders)}')
print('First order:', orders[0])

Total orders: 7
First order: {'order_id': 1, 'customer': 'Alice', 'amount': 250.0, 'status': 'completed'}


In [6]:
# Filter + write — keep only completed orders
completed = [o for o in orders if o['status'] == 'completed']
print(f'Completed: {len(completed)} of {len(orders)}')

with open('completed_orders.json', 'w') as f:
    json.dump(completed, f, indent=2)

# Verify
with open('completed_orders.json') as f:
    print(f.read())

Completed: 4 of 7
[
  {
    "order_id": 1,
    "customer": "Alice",
    "amount": 250.0,
    "status": "completed"
  },
  {
    "order_id": 3,
    "customer": "Carol",
    "amount": 450.0,
    "status": "completed"
  },
  {
    "order_id": 5,
    "customer": "Eve",
    "amount": 320.0,
    "status": "completed"
  },
  {
    "order_id": 7,
    "customer": "Grace",
    "amount": 180.0,
    "status": "completed"
  }
]


In [ ]:
# json.loads — parse a JSON *string* (common with API responses)
raw = '{"event": "user_signup", "user_id": 42, "ts": "2024-01-10"}'
event = json.loads(raw)
print(type(event), event)
print('user_id:', event['user_id'])

# json.dumps — Python object → JSON string
print(json.dumps({'status': 'ok', 'count': 7}, indent=2))

In [ ]:
# JSONL — one JSON object per line (common in DE pipelines)
events = [
    {'event': 'click', 'page': '/home',    'uid': 1},
    {'event': 'click', 'page': '/pricing', 'uid': 2},
    {'event': 'signup','page': '/register','uid': 3},
]

# Write JSONL
with open('events.jsonl', 'w') as f:
    for e in events:
        f.write(json.dumps(e) + '\n')

# Read JSONL
with open('events.jsonl') as f:
    records = [json.loads(line) for line in f]

print('JSONL records:', records)

---
## 3. TXT / Log Files — Line-by-Line and Generators

In [ ]:
# readlines() — loads all lines into a list (fine for small files)
with open('app.log') as f:
    lines = f.readlines()      # list of strings, each ending in '\n'

print(f'Total lines: {len(lines)}')
print('First line:', lines[0].strip())

In [ ]:
# Iterate line by line — O(1) memory, file never fully loaded
print('ERROR lines in app.log:')
with open('app.log') as f:
    for line in f:
        line = line.strip()
        if 'ERROR' in line:
            print(' ', line)

In [ ]:
# Generator — yields one ERROR line at a time; safe for 10 GB log files
def error_lines(path):
    with open(path) as f:
        for line in f:
            if 'ERROR' in line:
                yield line.strip()

# Usage — generator is lazy; nothing read until we iterate
errors = list(error_lines('app.log'))
print(f'{len(errors)} errors found:')
for e in errors:
    print(' ', e)

In [ ]:
# Write errors to a separate file
with open('errors.log', 'w') as out:
    for err in error_lines('app.log'):
        out.write(err + '\n')

with open('errors.log') as f:
    print('errors.log contents:\n', f.read())

In [ ]:
# Chunk generator — yield N rows at a time (common DE pattern for large CSVs)
def csv_chunks(path, chunk_size=3):
    with open(path, newline='') as f:
        reader = csv.DictReader(f)
        chunk = []
        for row in reader:
            chunk.append(row)
            if len(chunk) == chunk_size:
                yield chunk
                chunk = []
        if chunk:
            yield chunk

print('Processing employees.csv in chunks of 3:')
for i, chunk in enumerate(csv_chunks('employees.csv'), 1):
    print(f'  Chunk {i}: {[r["name"] for r in chunk]}')

---
## Quick Reference

| Task | Code |
|------|------|
| Read CSV as dicts | `csv.DictReader(f)` |
| Write CSV rows | `csv.writer(f).writerows(rows)` |
| Parse JSON file | `json.load(f)` |
| Parse JSON string | `json.loads(s)` |
| Write JSON | `json.dump(obj, f, indent=2)` |
| Read file line by line | `for line in f: line.strip()` |
| Generator for large file | `yield row` inside `with open(...)` |
| Group + aggregate from CSV | `defaultdict` + `csv.DictReader` |